In [1]:
import pandas as pd
import numpy as np

In [5]:
df = pd.read_parquet('combined.parquet')

In [6]:
df.columns

Index(['pickup_datetime', 'dropoff_datetime', 'ratecodeid', 'pulocationid',
       'dolocationid', 'passenger_count', 'trip_distance', 'fare_amount',
       'extra', 'mta_tax', 'tip_amount', 'tolls_amount',
       'improvement_surcharge', 'total_amount', 'payment_type', 'trip_type',
       'congestion_surcharge', 'is_green_ride', 'airport_fee'],
      dtype='object')

In [7]:
df['pickup_minute'] = df['pickup_datetime'].dt.minute

/tmp/ipykernel_2642907/210473013.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['pickup_datetime'] = pd.to_datetime(df['pickup_datetime'], errors='coerce')


In [8]:
df.drop(columns=["pickup_datetime", "dropoff_datetime", "cloudcover"], inplace=True, errors="ignore")
df["fare_amount"] = df["fare_amount"].str.replace(',', '').astype(float)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 39937688 entries, 0 to 39937687
Data columns (total 18 columns):
 #   Column                 Dtype  
---  ------                 -----  
 0   ratecodeid             float64
 1   pulocationid           int64  
 2   dolocationid           int64  
 3   passenger_count        float64
 4   trip_distance          object 
 5   fare_amount            float64
 6   extra                  float64
 7   mta_tax                float64
 8   tip_amount             float64
 9   tolls_amount           float64
 10  improvement_surcharge  float64
 11  total_amount           object 
 12  payment_type           float64
 13  trip_type              float64
 14  congestion_surcharge   float64
 15  is_green_ride          bool   
 16  airport_fee            float64
 17  pickup_minute          int32  
dtypes: bool(1), float64(12), int32(1), int64(2), object(2)
memory usage: 4.9+ GB


In [9]:
# Clean and split preciptype into lists
df['precip_list'] = df['preciptype'].apply(lambda x: [] if pd.isna(x) else x.split(','))

# Multi-hot encode into rain/snow columns
encoded = (
    df['precip_list']
    .explode()
    .pipe(pd.get_dummies)
    .groupby(level=0)
    .sum()
    .reindex(df.index, fill_value=0)
    .astype(bool)
)

# Ensure both 'rain' and 'snow' columns exist even if missing in data
for col in ['rain', 'snow']:
    if col not in encoded.columns:
        encoded[col] = False

KeyError: 'preciptype'

In [ ]:
# Merge back and drop intermediate columns
df.drop(columns=['precip_list', "preciptype", "snow", "rain"], inplace=True, errors="ignore")
df.columns

In [ ]:
df = pd.concat([df, encoded[['rain', 'snow']]], axis=1)

In [ ]:
df.columns

In [ ]:
df.info()

In [ ]:
df["total_amount"] = df["total_amount"].str.replace(',', '').astype(float)
df.info()

In [ ]:
df["payment_type"].isna().sum()
df = df.dropna(subset=["payment_type"])

In [ ]:
df["tip_percent"] = np.where(
    df["fare_amount"] > 0,
    (df["tip_amount"] / df["fare_amount"]) * 100,
    np.nan
)

In [ ]:
q1 = df["tip_percent"].quantile(0.33)
q2 = df["tip_percent"].quantile(0.67)

def tip_class(x):
    if x < q1:
        return 0 #low
    elif x < q2:
        return 1 #median
    else:
        return 2 # high

df["tip_category"] = df["tip_percent"].apply(tip_class)


In [ ]:
df.columns

In [ ]:
df.drop(columns=["total_amount", "tip_percent"], inplace=True, errors="ignore")

In [ ]:
df.describe()

In [ ]:
#remove unrealistic data
df = df[(df["fare_amount"] > 2) & (df["fare_amount"] < 1000)]
df = df[(df["trip_distance"] > 0.1) & (df["trip_distance"] < 50)]

In [ ]:
from sklearn.preprocessing import RobustScaler

scaler = RobustScaler()
num_cols = ["trip_distance", "fare_amount", "temp", "humidity", "windspeed"]
df[num_cols] = scaler.fit_transform(df[num_cols])


In [ ]:
df.isna().sum().sort_values(ascending=False)

In [ ]:
df.to_parquet("combined.parquet")

In [ ]:
df.describe()